# 07. LiRA null check
What catch rate does LiRA report when the models **never saw** the 'members'? We train fresh models on a different group of patients (the shadow data) and attack them with the same LiRA setup, still calling target_train patients 'members' and target_test patients 'non-members'. Nobody in either group was used, so the true answer is chance. Anything above 1% at 1% false alarms is the attack's own bias.
Trains new models only (about 25 minutes). Needs notebooks 01 to 05b. Finished models are skipped.

In [ ]:
import os, sys, json, gc, time, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = os.environ.get("PA_ROOT", "/content/drive/MyDrive/privacy-audit/")
sys.path.insert(0, ROOT)          # pa_lib.py must sit in the privacy-audit folder on Drive
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip('opacus')
from pa_lib import *
make_dirs()
print(versions())


Mounted at /content/drive
{'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'art': None, 'sklearn': '1.6.1'}


In [ ]:
from sklearn.metrics import roc_auc_score
Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
Xs, ys = load_split("shadow_train")                 # patients that appear in neither the members nor the non-members
input_dim = Xtr.shape[1]; n_tr = len(Xtr)
pool_X = np.concatenate([Xtr, Xte]); pool_y = np.concatenate([ytr, yte]); N = len(pool_y); idx = np.arange(N)
labels = np.r_[np.ones(n_tr), np.zeros(N - n_tr)]
EARLY = get_config()["early_epochs"]
NULL = MODELS + "null/"
SPECS = {"baseline_e50": ("plain", EPOCHS, 0.0), "baseline_early": ("plain", EARLY, 0.0)}
SPECS.update({f"dpsgd_eps{e}": ("dp", e, 0.0) for e in EPSILONS})
SPECS.update({f"baseline_wd{w:g}": ("plain", EPOCHS, w) for w in (1e-3, 1e-2, 1e-1)})
def lira_set(cfg):
    fs = sorted(f for f in os.listdir(LIRA + cfg) if f.endswith(".npz"))
    d = [np.load(LIRA + cfg + "/" + f) for f in fs]
    return np.stack([x["phi"] for x in d]), np.stack([x["mask"] for x in d])
print("rows used to train each null model:", n_tr, "| shadow rows available:", len(Xs))


rows used to train each null model: 5969 | shadow rows available: 7454


In [ ]:
RUN = list(SPECS)            # trim to run in chunks
t0 = time.time()
for cfg in RUN:
    kind, arg, wd = SPECS[cfg]
    for seed in SEEDS:
        if os.path.exists(f"{NULL}{cfg}_seed{seed}.pt"):
            continue
        sel = np.random.RandomState(100 + seed).choice(len(Xs), n_tr, replace=False)     # same size as the real training set
        if kind == "plain":
            m, meta = train_plain(Xs[sel], ys[sel], arg, seed, weight_decay=wd)
        else:
            m, meta = train_dp(Xs[sel], ys[sel], arg, seed)
        save_model(m, cfg, seed, meta, folder=NULL)
        del m; gc.collect()
    print(f"{cfg}: null models done ({(time.time() - t0) / 60:.1f} min)")


baseline_e50: null models done (0.7 min)
baseline_early: null models done (0.8 min)
dpsgd_eps0.5: null models done (2.6 min)
dpsgd_eps1: null models done (4.4 min)
dpsgd_eps2: null models done (6.3 min)
dpsgd_eps5: null models done (8.3 min)
dpsgd_eps10: null models done (10.7 min)
dpsgd_eps100: null models done (19.7 min)
baseline_wd0.001: null models done (20.2 min)
baseline_wd0.01: null models done (20.7 min)
baseline_wd0.1: null models done (21.3 min)


In [ ]:
rows = []
for cfg in SPECS:
    if not os.path.isdir(LIRA + cfg):
        print("skipping", cfg, "(no practice models)")
        continue
    phis, masks = lira_set(cfg)
    for seed in SEEDS:
        z = get_logits(load_model(cfg, seed, input_dim, folder=NULL), pool_X)
        s = lira_scores(z[idx, pool_y] - z[idx, 1 - pool_y], phis, masks)
        rows.append({"model": cfg, "seed": seed, "null_auc": roc_auc_score(labels, s),
                     "null_tpr@1%": tpr_at_fpr(labels, s, 0.01), "null_tpr@0.1%": tpr_at_fpr(labels, s, 0.001)})
nl = pd.DataFrame(rows)
lr = pd.read_csv(RESULTS + "lira_results.csv")
real = lr.groupby("model")[["auc", "tpr@1%fpr"]].mean().rename(columns={"auc": "real_auc", "tpr@1%fpr": "real_tpr@1%"})
table = nl.groupby("model")[["null_auc", "null_tpr@1%", "null_tpr@0.1%"]].mean().join(real).reindex([c for c in SPECS if c in set(nl["model"])]).round(3)
nl.to_csv(RESULTS + "lira_null_check_all_seeds.csv", index=False); table.to_csv(RESULTS + "lira_null_check.csv")
print(table.to_string())
print("\nnull columns: LiRA on models that never saw these patients (the attack's own baseline; ideally AUC 0.5 and catch rate 0.010).")
print("real columns: LiRA on the real models. A real value clearly above its null value means a real leak.")


                  null_auc  null_tpr@1%  null_tpr@0.1%  real_auc  real_tpr@1%
model                                                                        
baseline_e50         0.492        0.012          0.003     0.639        0.074
baseline_early       0.498        0.014          0.003     0.534        0.025
dpsgd_eps0.5         0.501        0.013          0.002     0.501        0.015
dpsgd_eps1           0.502        0.013          0.002     0.502        0.015
dpsgd_eps2           0.503        0.013          0.002     0.506        0.016
dpsgd_eps5           0.502        0.013          0.002     0.509        0.016
dpsgd_eps10          0.501        0.013          0.003     0.511        0.017
dpsgd_eps100         0.501        0.012          0.003     0.520        0.019
baseline_wd0.001     0.491        0.009          0.002       NaN          NaN
baseline_wd0.01      0.498        0.013          0.002       NaN          NaN
baseline_wd0.1       0.504        0.020          0.008       NaN